[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dxnn1u/slides/blob/main/decks/pytorch-basic/lecture6_lab_inference.ipynb)

# [ Lecture 6 ] Lab — Make your MLP fast

In the lecture we **trained** an MLP on MNIST. Now we **use** it: run inference on all 10,000 test images,
and make that as fast as possible.

This is how the AI tasks in student HPC competitions work: you get a model, a fixed machine,
and a quality bar, and you are scored on speed.

### Rules
1. Predict the digit for **all 10,000 test images**.
2. Start from the test set in **CPU memory** (`X_cpu`, `X_raw` or `X_pinned`, prepared below) and return the predictions as a **CPU tensor**.
3. **Accuracy ≥ 97%.**
4. Score = **samples / second** (higher is better). Training and one-time model setup are not timed.

### Levels
| Level | Idea |
|---|---|
| 0 · Measure correctly | GPU calls are asynchronous: synchronize, warm up, repeat |
| 1 · Inference mode ✏️ | no graph for backward is needed |
| 2 · Move data less ✏️ | one big copy, no `.item()` per sample |
| 3 · Batch size | throughput vs. latency |
| 4 · Lower precision ✏️ | fp16, then check the accuracy |
| 5 · `torch.compile` | fewer, bigger GPU operations |
| 6 · Profiler | find where the time goes |
| ★ · Challenge ✏️ | your own tricks; post your samples / s |

✏️ = you write code. Each **TODO** has a hidden solution cell right below it: double-click it if you are stuck.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. Levels 4–5 need a GPU.

## Setup

In [ ]:
import copy
import math
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
cc = torch.cuda.get_device_capability(0) if device == "cuda" else (0, 0)
gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU only"

print("PyTorch:", torch.__version__)
print("Device :", gpu_name, f"(compute capability {cc[0]}.{cc[1]})" if device == "cuda" else "")
if device == "cpu":
    print("⚠ No GPU: Levels 0–3 still run, Levels 4–5 are skipped. Runtime → Change runtime type → T4 GPU.")
elif cc < (7, 0):
    print("⚠ This GPU is too old for torch.compile (Level 5). Switch the runtime to T4 GPU.")

## Step 1 · Train the model
Same model and training loop as the lecture (3 epochs, about a minute on a GPU).

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)            # (B, 1, 28, 28) -> (B, 784)
        x = self.relu(self.fc1(x))     # (B, 256)
        x = self.relu(self.fc2(x))     # (B, 128)
        return self.fc3(x)             # (B, 10) logits

transform = transforms.Compose([
    transforms.ToTensor(),                        # uint8 0–255 -> float32 0–1
    transforms.Normalize((0.1307,), (0.3081,)),   # MNIST mean / std
])
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

torch.manual_seed(0)
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

for epoch in range(3):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch + 1} | last loss {loss.item():.4f}")

## Step 2 · The test set as tensors
Instead of a `Dataset` that decodes one image at a time, we keep the whole test set as tensors in CPU memory.
These are the starting points the rules allow:

| name | shape | dtype | what |
|---|---|---|---|
| `X_raw` | (10000, 28, 28) | uint8 | raw pixels 0–255 (you normalize them yourself) |
| `X_cpu` | (10000, 1, 28, 28) | float32 | already `ToTensor` + `Normalize`d |
| `X_pinned` | same as `X_cpu` | float32 | `X_cpu` in pinned memory (GPU only; see Level 2) |

In [ ]:
raw_test = datasets.MNIST(root="./data", train=False, download=True)
X_raw = raw_test.data                    # (10000, 28, 28) uint8
y_true = raw_test.targets                # (10000,) int64
N = len(y_true)

MEAN, STD = 0.1307, 0.3081
X_cpu = ((X_raw.float() / 255 - MEAN) / STD).unsqueeze(1)    # same as ToTensor + Normalize
X_pinned = X_cpu.pin_memory() if device == "cuda" else X_cpu

print("X_raw:", tuple(X_raw.shape), X_raw.dtype, f"{X_raw.numel() * X_raw.element_size() / 1e6:.1f} MB")
print("X_cpu:", tuple(X_cpu.shape), X_cpu.dtype, f"{X_cpu.numel() * X_cpu.element_size() / 1e6:.1f} MB")

## Step 3 · The scoreboard
`report(name, predict)` times a `predict()` function that follows the rules, checks its accuracy,
and adds it to the scoreboard. Level 0 explains why `benchmark` looks the way it does.

In [ ]:
def sync():
    # Wait until the GPU has finished all queued work (no-op on CPU).
    if device == "cuda":
        torch.cuda.synchronize()

def benchmark(fn, repeat=5, warmup=2):
    for _ in range(warmup):          # first calls include one-time setup
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(repeat):
        out = fn()
    sync()                           # make sure the GPU is really done
    return (time.perf_counter() - t0) / repeat, out

results = {}   # name -> (seconds, samples/s, accuracy, predict)

def report(name, predict, repeat=5, warmup=2):
    sec, preds = benchmark(predict, repeat, warmup)
    assert preds.device.type == "cpu" and preds.shape == (N,), "return CPU predictions for all images"
    acc = (preds == y_true).float().mean().item()
    tput = N / sec
    results[name] = (sec, tput, acc, predict)
    base = results.get("L0 baseline")
    speedup = f"{tput / base[1]:7.1f}× vs. baseline" if base else ""
    flag = "" if acc >= 0.97 else "  ✗ accuracy below 97%"
    print(f"{name:<22} {sec * 1e3:9.2f} ms  {tput:>12,.0f} samples/s  acc {acc:.2%}  {speedup}{flag}")

## Level 0 · Measure correctly
PyTorch GPU calls are **asynchronous**: `a @ a` only *queues* the work on the GPU and returns right away.
If you stop the clock without waiting, you measure the time to *launch* the work, not to *do* it.

So every timing in this lab:
1. **warms up** first (the first calls include one-time setup),
2. calls `torch.cuda.synchronize()` before reading the clock,
3. **repeats** and averages.

In [ ]:
n = 4096 if device == "cuda" else 1024
a = torch.randn(n, n, device=device)
a @ a; sync()                                    # warm-up

t0 = time.perf_counter(); b = a @ a
t_launch = time.perf_counter() - t0
sync()

t0 = time.perf_counter(); b = a @ a; sync()
t_real = time.perf_counter() - t0

print(f"without synchronize: {t_launch * 1e3:8.3f} ms  <- only the launch")
print(f"with synchronize   : {t_real * 1e3:8.3f} ms  <- the real time")

### The baseline
A straightforward first attempt. It is correct, but slow. Each later level fixes one thing.

In [ ]:
loader64 = DataLoader(TensorDataset(X_cpu), batch_size=64)
model.train()                                # as we left it after training

def predict_v0():
    preds = []
    for (xb,) in loader64:
        xb = xb.to(device)                   # 157 small copies to the GPU
        logits = model(xb)                   # autograd records a graph for backward
        for p in logits.argmax(dim=1):
            preds.append(p.item())           # one .item() per image
    return torch.tensor(preds)

report("L0 baseline", predict_v0, repeat=2, warmup=1)

## Level 1 · Inference mode
For inference we only run forward, so we do not need the graph autograd builds for `backward()`.
- `model.eval()`: switch layers like Dropout / BatchNorm to inference behaviour. Our MLP has none, but always do it.
- `torch.inference_mode()`: like `torch.no_grad()`, and also skips some bookkeeping autograd would need later.
  It works as a `with` block or as a decorator (`@torch.inference_mode()` on the line above `def`).

> **TODO:** make `predict_v1` run without autograd. One line is enough.

If the speed-up is small, that is a clue: the time is going somewhere else. Level 2 finds it.

In [ ]:
def check_no_grad(predict):
    # Records whether autograd was on while the model ran.
    seen = []
    hook = model.register_forward_hook(lambda m, i, o: seen.append(torch.is_grad_enabled()))
    try:
        predict()
    finally:
        hook.remove()
    print("✓ autograd is off" if seen and not any(seen) else "✗ autograd is still recording")

model.eval()

# TODO: turn off autograd for this function
def predict_v1():
    preds = []
    for (xb,) in loader64:
        logits = model(xb.to(device))
        for p in logits.argmax(dim=1):
            preds.append(p.item())
    return torch.tensor(preds)

check_no_grad(predict_v1)
report("L1 inference_mode", predict_v1, repeat=2, warmup=1)

In [ ]:
#@title 💡 Level 1 solution (double-click to show the code; running it replaces your version)
@torch.inference_mode()
def predict_v1():
    preds = []
    for (xb,) in loader64:
        logits = model(xb.to(device))
        for p in logits.argmax(dim=1):
            preds.append(p.item())
    return torch.tensor(preds)

check_no_grad(predict_v1)
report("L1 inference_mode", predict_v1, repeat=2, warmup=1)

## Level 2 · Move data less
`.item()` copies **one number** from the GPU to the CPU, and the CPU has to **wait** for the GPU to finish first.
The baseline does that 10,000 times. It also makes 157 small copies to the GPU.

> **TODO:** write `predict_v2` so that it
> 1. copies the whole `X` to the GPU **once** (`X.to(device, non_blocking=True)`),
> 2. runs the model batch by batch on slices `X[i:i + batch_size]`, keeping each batch's `argmax` **on the GPU** in a list,
> 3. joins the list with `torch.cat` and copies the result back to the CPU **once** (`.cpu()`).
>
> Levels 3–5 reuse this function, so make it work before moving on.

In [ ]:
@torch.inference_mode()
def predict_v2(batch_size=64, X=X_cpu):
    # TODO
    raise NotImplementedError("Level 2: write predict_v2")

report("L2 move data less", predict_v2)

In [ ]:
#@title 💡 Level 2 solution (double-click to show the code; running it replaces your version)
@torch.inference_mode()
def predict_v2(batch_size=64, X=X_cpu):
    X = X.to(device, non_blocking=True)          # one big copy
    preds = []
    for i in range(0, N, batch_size):
        preds.append(model(X[i:i + batch_size]).argmax(dim=1))   # stays on the GPU
    return torch.cat(preds).cpu()                # one copy back, one wait

report("L2 move data less", predict_v2)

### Pinned memory
Normal (*pageable*) CPU memory has to be staged through a pinned buffer before the GPU can read it.
Memory that is already **pinned** skips that step, and `non_blocking=True` lets the copy run in the background.

In [ ]:
if device == "cuda":
    mb = X_cpu.numel() * X_cpu.element_size() / 1e6
    for name, X in [("pageable", X_cpu), ("pinned", X_pinned)]:
        sec, _ = benchmark(lambda X=X: X.to(device, non_blocking=True), repeat=20)
        print(f"{name:<9} copy: {sec * 1e3:6.2f} ms  ({mb / 1e3 / sec:5.1f} GB/s)")
    report("L2 + pinned", lambda: predict_v2(64, X_pinned))
else:
    print("needs a GPU")

## Level 3 · Batch size
A bigger batch gives the GPU more work per call, so fewer calls are needed. Two numbers matter:
- **Throughput**: images per second. What we are scored on.
- **Latency**: how long one batch takes. What a user waiting for one answer feels.

Run the sweep and look at both plots.

In [ ]:
X_best = X_pinned
sizes = [16, 64, 256, 1024, 4096, N]
tputs, lats = [], []
for bs in sizes:
    sec, _ = benchmark(lambda bs=bs: predict_v2(bs, X_best))
    tputs.append(N / sec)
    lats.append(sec / math.ceil(N / bs) * 1e3)    # ≈ ms per batch
    print(f"batch {bs:>6}: {N / sec:>12,.0f} samples/s   ≈ {lats[-1]:7.3f} ms per batch")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(sizes, tputs, "o-"); ax1.set_title("Throughput (higher is better)"); ax1.set_ylabel("samples / s")
ax2.plot(sizes, lats, "o-", color="tab:red"); ax2.set_title("Latency per batch (lower is better)"); ax2.set_ylabel("ms")
for ax in (ax1, ax2):
    ax.set_xscale("log"); ax.set_xlabel("batch size")
plt.tight_layout(); plt.show()

BEST_BS = sizes[tputs.index(max(tputs))]
report(f"L3 batch {BEST_BS}", lambda: predict_v2(BEST_BS, X_best))

**Questions**
1. Throughput grows with batch size at first. Why does it stop growing?
2. A chat app answers one user at a time, and the user is waiting. Would you still pick `BEST_BS` there? Why?

## Level 4 · Lower precision
fp16 halves the bytes per number, and GPUs have special units (Tensor Cores) that multiply fp16 matrices much faster.
The price is precision, so **always re-check the accuracy**.

Which half-precision type you can use depends on the hardware: the T4 (compute capability 7.5) has no native bf16;
that needs compute capability 8.0 or newer (A100, H100, RTX 30xx…).

Here we convert the whole model with `.half()`. For bigger models, `torch.autocast` is the usual way:
it picks fp16 for the operations that are safe and keeps fp32 for the rest.

> **TODO:** write `predict_v4`: the same as `predict_v2`, but
> - use `net` (the fp16 copy of the model) instead of `model`,
> - convert the input to fp16 with `.half()` after copying it to the GPU,
> - start from `X_best`.

In [ ]:
if device == "cuda":
    print("native bf16:", cc >= (8, 0))
    model_fp16 = copy.deepcopy(model).half()

    @torch.inference_mode()
    def predict_v4(batch_size=BEST_BS, net=model_fp16):
        # TODO
        raise NotImplementedError("Level 4: write predict_v4")

    report("L4 fp16", predict_v4)
else:
    print("needs a GPU")

In [ ]:
#@title 💡 Level 4 solution (double-click to show the code; running it replaces your version)
if device == "cuda":
    @torch.inference_mode()
    def predict_v4(batch_size=BEST_BS, net=model_fp16):
        X = X_best.to(device, non_blocking=True).half()
        preds = [net(X[i:i + batch_size]).argmax(dim=1) for i in range(0, N, batch_size)]
        return torch.cat(preds).cpu()

    report("L4 fp16", predict_v4)

**Questions**
1. Did the accuracy change? Could it, in principle?
2. Try calling `.half()` **before** the copy (on the CPU) instead of after. Which is faster, and why?

## Level 5 · `torch.compile`
Our forward pass is a chain of small GPU operations: matmul, add bias, ReLU, matmul, …
Each one is a separate **kernel** (a function that runs on the GPU), and each launch costs a few microseconds.
`torch.compile` traces the model and generates fewer, fused kernels.
`mode="reduce-overhead"` also records the whole sequence once as a **CUDA graph** and replays it with a single launch.

The first calls are slow because they compile; `benchmark` warms up first.
Launch overhead matters most when each kernel is small, so also try a small batch size.

In [ ]:
if device == "cuda" and cc >= (7, 0):
    try:
        compiled = torch.compile(model_fp16, mode="reduce-overhead")
        report("L5 compile", lambda: predict_v4(BEST_BS, compiled), warmup=3)
        report("L4 fp16, batch 256", lambda: predict_v4(256))
        report("L5 compile, batch 256", lambda: predict_v4(256, compiled), warmup=3)
    except Exception as e:
        print("torch.compile failed:", type(e).__name__, e)
else:
    print("needs a GPU with compute capability ≥ 7.0")

**Question:** compare the gain from `torch.compile` at batch `BEST_BS` and at batch 256. Where does it help more, and why?

## Level 6 · Profiler
Before optimizing, find out **where the time goes**. `torch.profiler` records every operation.

Compare the baseline with your best version so far:
- Which operation takes the most time in the baseline? Which level removed it?
- What dominates now?

In [ ]:
from torch.profiler import ProfilerActivity, profile

activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device == "cuda" else [])
valid = {k: v for k, v in results.items() if v[2] >= 0.97}
best_name = max(valid, key=lambda k: valid[k][1])

for name in ["L0 baseline", best_name]:
    predict = results[name][3]
    predict(); sync()                           # warm up outside the profiler
    with profile(activities=activities) as prof:
        predict(); sync()
    print(f"===== {name} =====")
    print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=8))

## ★ Challenge
Beat your best score. Same rules: start from `X_cpu` / `X_raw` / `X_pinned`, return CPU predictions, accuracy ≥ 97%.

Ideas (no solutions here, that is the fun part):
- **Send fewer bytes.** `X_raw` is uint8: 4× smaller than float32. Copy that and normalize on the GPU.
- **Fold `Normalize` into `fc1`.** `(x − mean) / std` is linear, and so is `fc1`. Can you change `fc1.weight` and `fc1.bias` once, so the model accepts `x / 255` (or even raw pixels) directly?
- **Overlap copy and compute.** While the GPU works on chunk *k*, copy chunk *k + 1* (pinned memory, `non_blocking=True`, CUDA streams).
- **Combine levels.** Batch size × fp16 × compile: the best batch size may change once the others are on.

In [ ]:
@torch.inference_mode()
def my_predict():
    # Your version here. Start from X_cpu / X_raw / X_pinned, return CPU predictions.
    return predict_v2(BEST_BS, X_best)

report("my best", my_predict)

### Scoreboard

In [ ]:
print(f"GPU: {gpu_name} | PyTorch {torch.__version__}\n")
base = results["L0 baseline"][1]
for name, (_, tput, acc, _) in sorted(results.items(), key=lambda kv: -kv[1][1]):
    flag = "" if acc >= 0.97 else "  ✗ acc < 97%"
    print(f"{name:<22} {tput:>12,.0f} samples/s  {tput / base:8.1f}×  acc {acc:.2%}{flag}")

**Submit:** post your GPU name, your best samples / s, its accuracy, and one sentence on the trick that helped most.